# Проверка обработки ЭКГ на реальных данных

Разбор отдельных сбоев с графиками до/после: [шесть примеров](epi_ecg_failure_examples.ipynb).

Сравнение для 20 записей из предыдущей таблицы: **400 фиксированных окон по 300 с**, по 20 на запись. Кандидаты построены с шагом 5 с и параметрами разметки исходного ноутбука (lead 300–900 с); из них равномерно выбраны окна независимо от качества. Это контроль обработки, не обучающая выборка и не повторение прежних 1258 окон. Точные интервалы сохранены в `window_manifest.csv`.

Исходный каталог и `.e` читаются без изменения. Числа ниже получены скриптом `scripts/compare_ecg_preprocessing.py`. Можно выполнить все клетки: по умолчанию пересчёт 400 окон и обучение моделей не запускаются.

**Показатель сети:** `P(48–52 Гц) / P(0.5–40 Гц)`, Welch с отрезками 4 с; края по 2 с исключены только из оценки спектра. Меньше — меньше относительная мощность сети. Это новое явное определение, не формула `line_noise_ratio` из недоступной версии epi 1.3.0. Снижение этого показателя само по себе не доказывает пригодность ЭКГ.

Оба фильтра получают одинаково прочитанные окна. Чтение, фильтрация и отбраковка показаны отдельно. Все приведённые медианы до/после вычислены **до отбраковки**, поэтому улучшение нельзя объяснить удалением плохих окон.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Image

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "epi/__init__.py").exists())
sys.path.insert(0, str(ROOT))
OUT = ROOT / "data/reports/preprocessing_v2_20261010"
RECOMPUTE = False  # True: заново прочитать те же окна; несколько минут, без обучения.
if RECOMPUTE:
    from scripts.compare_ecg_preprocessing import main
    main()

table = pd.read_csv(OUT / "comparison_by_patient.csv")
windows = pd.read_csv(OUT / "window_comparison.csv")
reader_check = pd.read_csv(OUT / "reader_validation.csv")
assert len(windows) == int(table.n.sum())
print(f"Записей: {len(table)}; окон: {len(windows)}")
print(f"QC после правильного чтения: раньше {windows.keep_before_on_correct_read.sum()}, теперь {windows.keep_after.sum()}")
print("Старый ридер: ошибок чтения", (~windows.legacy_read_ok).sum(),
      "; принятых старым QC", windows.legacy_keep.sum())
print("Проверок с Neo:", len(reader_check), "; расхождений до:",
      (reader_check.before_max_error_uv > 1e-6).sum(), "; после:",
      (reader_check.after_max_error_uv > 1e-6).sum())


## Таблица до/после

`raw_median` и `raw_p95` — медиана и 95-й процентиль мощности сети в исходных окнах после приведения к 256 Гц. `clean_before` — прежний узкий notch 50 Гц и полоса 0.5–40 Гц. `clean_after` — подавление 48–52 Гц, отражение краёв и та же полоса 0.5–40 Гц. `noise_reduction_factor = clean_before / clean_after`.

`n_keep_before` — прежняя политика QC на правильно прочитанных окнах; `n_keep_after` — новая политика. Отдельная колонка `legacy_read_errors` показывает ошибки старого чтения. Снижение числа допущенных окон — отбраковка, а не восстановление их сигнала. Ноль принятых означает только эти 20 окон, а не приговор всей записи или пациенту.


In [ ]:
display(table[["patient", "recording_id", "n", "raw_median", "raw_p95",
               "clean_before", "clean_after", "noise_reduction_factor",
               "n_keep_before", "n_keep_after", "legacy_read_errors"]].style.format({
    key: "{:.3g}" for key in ["raw_median", "raw_p95", "clean_before", "clean_after", "noise_reduction_factor"]
}))


## Какие окна исключены

Проверяются нули/нечисловые значения, плато дольше 0.5 с, доля отсчётов на пределах АЦП более 1% (до ресэмплинга) и выбросы после фильтра. Для выброса одновременно нужны пик более 1000 мкВ и более чем в 8 раз выше медианы секундных пиков этого окна. Так регулярные высокие R-зубцы не исключаются только из-за амплитуды. Пороги — проверяемые эвристики, не экспертная разметка качества.

В этом сравнении применены фиксированные правила. Дополнительные обучаемые пороги исходного модельного ноутбука должны оцениваться отдельно внутри каждого train-фолда; они могут исключить ещё окна. Числа ниже не измеряют точность TCN.


In [ ]:
display(windows.groupby("label").agg(
    всего=("label", "size"), принято_раньше=("keep_before_on_correct_read", "sum"),
    принято_сейчас=("keep_after", "sum")))
display(windows.loc[~windows.keep_after, "reasons"].str.split(";").explode().value_counts().rename("окон"))
display(windows.loc[~windows.keep_after, ["patient", "recording_id", "segment", "start_sec",
    "label", "reasons", "native_adc_rail_fraction", "peak_uv", "peak_ratio"]].head(20))


## Проверенные примеры сигнала

На первых двух примерах фильтр не восстанавливает выброс и насыщение — новые правила исключают эти окна. Следующие четыре примера с видимой сердечной структурой сохранены. Это небольшая проверка конкретных дефектов на реальных данных, не оценка чувствительности и специфичности QC на всех пациентах.


In [ ]:
display(pd.read_csv(OUT / "reviewed_examples.csv"))
display(Image(filename=str(OUT / "signal_examples.png")))


## Посмотреть конкретное окно

Укажите `RECORDING_ID`, `SEGMENT`, `START_SEC`, `LENGTH_SEC`. Для примера выброса: 85, 0, 405, 10; для насыщения: 210, 0, 4475, 10. Полные 300-секундные интервалы и причины для таблицы лежат в `windows`. Зелёный/красный цвет показывает **решение QC**, а не класс приступа.


In [ ]:
import sqlite3
from fractions import Fraction
from scipy.signal import resample_poly
import matplotlib.pyplot as plt
from epi.reader import NicoletEReader
from epi.preprocessing import preprocess, preprocess_ecg, ecg_quality_report, ecg_raw_quality_report

RECORDING_ID, SEGMENT, START_SEC, LENGTH_SEC = 85, 0, 405.0, 10.0
with sqlite3.connect(f"file:{ROOT}/data/catalog.sqlite?mode=ro", uri=True) as conn:
    rel_path = conn.execute("SELECT rel_path FROM recordings WHERE id=?", (RECORDING_ID,)).fetchone()[0]
reader = NicoletEReader(ROOT / "data/epi_data" / rel_path, segment=SEGMENT, type_names={})
native = reader.read(["EKG"], START_SEC, LENGTH_SEC).T
source_qc = ecg_raw_quality_report(native, reader.sampling_rate, adc_limits=reader.channel_limits("EKG"))
fraction = Fraction(256 / reader.sampling_rate).limit_denominator(10000)
raw = resample_poly(native, fraction.numerator, fraction.denominator, axis=-1).astype("float32")
before, after = preprocess(raw, 256), preprocess_ecg(raw, 256)
qc = ecg_quality_report(raw, 256, filtered=after)
keep = source_qc["keep"] and qc["keep"]
reasons = sorted(set((source_qc["reasons"] + ";" + qc["reasons"]).split(";")) - {"ok"})
print("Принято:", keep, "; причины:", reasons or ["ok"])
t = START_SEC + np.arange(raw.shape[-1]) / 256
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True, layout="constrained")
for ax, x, title in zip(axes, [raw, before, after], ["Исходный сигнал, 256 Гц", "Прежний фильтр", "Новый фильтр"]):
    ax.plot(t, x[0], linewidth=.7)
    ax.set(title=title, ylabel="мкВ")
    ax.grid(alpha=.2)
axes[-1].set_facecolor("#edf6ee" if keep else "#fff0ef")
axes[-1].set_xlabel("Секунды от начала сегмента")
plt.show()


Для обучения маску `keep` применяют одинаково к `x`, `y`, `groups` и метаданным; нормализацию делают после QC. Сначала разделяют пациентов, затем подбирают любые обучаемые пороги только на train. После отбраковки обязательно пересчитывают размеры обоих классов. Положительные окна в этом ноутбуке соответствуют настройкам исходного эксперимента 300–900 с, а не обсуждавшемуся раньше диапазону 30–120 с.
